In [1]:
!pip install konlpy
from konlpy.tag import Okt
Okt().morphs("끝")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 14.6 MB/s eta 0:00:00


['끝']

In [2]:
!pip install konlpy

In [1]:
from konlpy.tag import Okt

okt = Okt()
okt.morphs("이제 진짜 끝났다")

['이제', '진짜', '끝났다']

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
base_path = '/content/drive/MyDrive/nlp_data'

train_path = base_path + '/ratings_train.txt'
test_path  = base_path + '/ratings_test.txt'
all_path   = base_path + '/ratings.txt'

In [5]:
import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/nlp_data/ratings_train.txt', sep='\t', encoding='utf-8')
train_df.head(3)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0


In [6]:
train_df['label'].value_counts()

,count
label,
0,75173
1,74827


In [11]:
import pandas as pd
import re

base_path = '/content/drive/MyDrive/nlp_data'

train_df = pd.read_csv(
    base_path + '/ratings_train.txt',
    sep='\t',
    encoding='utf-8'
)

test_df = pd.read_csv(
    base_path + '/ratings_test.txt',
    sep='\t',
    encoding='utf-8'
)

train_df = train_df.fillna(' ')
test_df = test_df.fillna(' ')

train_df['document'] = train_df['document'].apply(lambda x: re.sub(r'\d+', ' ', x))
test_df['document'] = test_df['document'].apply(lambda x: re.sub(r'\d+', ' ', x))

train_df.drop('id', axis=1, inplace=True)
test_df.drop('id', axis=1, inplace=True)

In [12]:
from konlpy.tag import Okt

twitter = Okt()

def tw_tokenizer(text):
    return twitter.morphs(text)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

tfidf_vect = TfidfVectorizer(
    tokenizer=tw_tokenizer,
    ngram_range=(1,1),
    min_df=5,
    max_df=0.9
)

tfidf_matrix_train = tfidf_vect.fit_transform(train_df['document'])

In [15]:
lg_clf = LogisticRegression(random_state=0, solver='liblinear')

params = {'C': [1, 3.5, 4.5, 5.5, 10]}

grid_cv = GridSearchCV(
    lg_clf,
    param_grid=params,
    cv=3,
    scoring='accuracy',
    verbose=1
)

grid_cv.fit(tfidf_matrix_train, train_df['label'])

print(grid_cv.best_params_, round(grid_cv.best_score_, 4))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
{'C': 3.5} 0.8498


In [16]:
from sklearn.metrics import accuracy_score

tfidf_matrix_test = tfidf_vect.transform(test_df['document'])

best_estimator = grid_cv.best_estimator_
preds = best_estimator.predict(tfidf_matrix_test)

print('Logistic Regression 정확도:', accuracy_score(test_df['label'], preds))

Logistic Regression 정확도: 0.84932
